# LeetCode 678 — Valid Parenthesis String bằng Python

Link đề: https://leetcode.com/problems/valid-parenthesis-string/ · Độ khó: **Medium** · Chủ đề: String, Stack (ngăn xếp), Greedy (tham lam), Dynamic Programming, Parentheses

Skeleton LeetCode:

```python
class Solution:
    def checkValidString(self, s: str) -> bool:
        ...
```

Ghi chú: bài kiểm tra chuỗi ngoặc quen thuộc, nhưng có thêm ký tự **`*` "đa năng"** (có thể là `(`, `)` hoặc rỗng). Nếu chỉ đoán thử từng khả năng thì nổ `3^k`; điểm hay của bài là nhận ra **tập các số ngoặc mở chưa đóng khả dĩ luôn là một khoảng liên tục** `[lo, hi]` nên chỉ cần theo dõi hai số. Notebook đi từ vét cạn -> ghi nhớ -> hai ngăn xếp -> khoảng `[lo, hi]` -> hai lượt quét.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chuỗi chỉ có `(`, `)`, `*`; `*` được chọn làm `(`, `)` hoặc rỗng; hỏi có **tồn tại** một cách chọn làm chuỗi ngoặc hợp lệ không.
2. Nắm các công cụ cần thiết: biến đếm "số ngoặc mở chưa đóng" (balance), ngăn xếp bằng `list` (`append`/`pop`), `set` các trạng thái khả dĩ, `functools.lru_cache`, duyệt ngược `s[::-1]`, `itertools.product` (cho oracle).
3. Hiểu vì sao bài ngoặc thường chỉ cần **một biến đếm** và vì sao khi có `*` trạng thái là **một tập** số đếm, rồi **thu gọn thành khoảng** `[lo, hi]`.
4. Đi qua 5 cách: đệ quy thử mọi lựa chọn `3^k`, đệ quy có nhớ `(i, balance)`, hai ngăn xếp chỉ số, tham lam khoảng `[lo, hi]`, hai lượt quét đếm; cùng một oracle quy hoạch động khoảng `O(n^3)` độc lập.
5. Tránh các lỗi kinh điển: chỉ so tổng số ký tự, quên chặn `lo >= 0`, quên dừng khi `hi < 0`, ghép `(` với `*` bỏ qua thứ tự, chỉ quét một lượt.

## 2. Đề bài & ví dụ

Cho chuỗi `s` chỉ gồm `'('`, `')'` và `'*'`. Trả về `true` nếu `s` **hợp lệ**.

Quy tắc hợp lệ:

- Mỗi `'('` phải có một `')'` tương ứng.
- Mỗi `')'` phải có một `'('` tương ứng.
- `'('` phải đứng **trước** `')'` tương ứng.
- `'*'` có thể được xem là một `')'`, hoặc một `'('`, hoặc chuỗi rỗng `""`.

**Ví dụ 1:** `s = "()"` -> `true`

**Ví dụ 2:** `s = "(*)"` -> `true` (`*` là rỗng: `"()"`)

**Ví dụ 3:** `s = "(*))"` -> `true` (`*` là `(`: `"(())"`)

**Ví dụ 4:** `s = "("` -> `false`

**Ràng buộc:**

- `1 <= s.length <= 100`
- `s[i]` là `'('`, `')'` hoặc `'*'`.

Lưu ý: ta chỉ cần **một** cách gán hợp lệ cho các dấu `*`, không cần đếm hay liệt kê.

In [1]:
from typing import List
from functools import lru_cache
from itertools import product
import random, time

examples = [
    ("()", True),
    ("(*)", True),
    ("(*))", True),
    ("(", False),
]
for s, expected in examples:
    print(f"s={s!r:8} len={len(s)}  kỳ vọng={expected}")

s='()'     len=2  kỳ vọng=True
s='(*)'    len=3  kỳ vọng=True
s='(*))'   len=4  kỳ vọng=True
s='('      len=1  kỳ vọng=False


## 3. Kiến thức nền

### 3.1. Chuỗi ngoặc không có `*`: một biến đếm là đủ

Gọi `bal` (balance) là số `(` đang chờ đóng. Gặp `(` thì `bal += 1`, gặp `)` thì `bal -= 1`. Chuỗi hợp lệ khi và chỉ khi **`bal` không bao giờ âm** (không có `)` thừa ở giữa) và **cuối cùng `bal == 0`** (không còn `(` thừa).

In [2]:
def valid_plain(t: str) -> bool:
    bal = 0
    for c in t:
        bal += 1 if c == "(" else -1
        if bal < 0:
            return False          # có ')' mà không có '(' trước đó
    return bal == 0

for t in ["()", "(())", "()()", ")(", "(()", "())", ""]:
    print(f"{t!r:8} -> {valid_plain(t)}")

'()'     -> True
'(())'   -> True
'()()'   -> True
')('     -> False
'(()'    -> False
'())'    -> False
''       -> True


### 3.2. Ngăn xếp bằng `list`

Ngăn xếp (stack) là kiểu "vào sau, ra trước": `append(x)` đẩy vào đỉnh, `pop()` lấy ra từ đỉnh. Trong Python dùng `list` là đủ. Ta sẽ cất **chỉ số** của `(` và `*` để biết thứ tự.

In [3]:
st = []
st.append(3); st.append(7); st.append(9)
print(st, "đỉnh =", st[-1])
print(st.pop(), st.pop(), st)
print("rỗng?", not st)
st.pop()
print("rỗng?", not st)

[3, 7, 9] đỉnh = 9
9 7 [3]
rỗng? False
rỗng? True


### 3.3. Tập các trạng thái khả dĩ

Khi gặp `*`, giá trị `bal` có **ba** khả năng: `bal - 1` (nếu `*` là `)`), `bal` (rỗng), `bal + 1` (nếu `*` là `(`). Thay vì chọn một, ta theo dõi **cả tập** `S` các `bal` khả dĩ sau mỗi ký tự. Quy tắc cập nhật:

- `(`: mọi phần tử `+1`.
- `)`: mọi phần tử `-1`, rồi **bỏ** các giá trị âm (nhánh đó đã vô hiệu).
- `*`: mỗi phần tử `b` thành `b - 1`, `b`, `b + 1`, rồi bỏ các giá trị âm.

Cuối cùng chuỗi hợp lệ nếu `0` thuộc `S`.

In [4]:
def reachable_sets(s: str):
    S = {0}
    history = [S]
    for c in s:
        if c == "(":
            S = {b + 1 for b in S}
        elif c == ")":
            S = {b - 1 for b in S if b - 1 >= 0}
        else:
            S = {x for b in S for x in (b - 1, b, b + 1) if x >= 0}
        history.append(S)
    return history

for step, S in enumerate(reachable_sets("(*))")):
    print(f"sau {step} ký tự: {sorted(S)}")

sau 0 ký tự: [0]
sau 1 ký tự: [1]
sau 2 ký tự: [0, 1, 2]
sau 3 ký tự: [0, 1]
sau 4 ký tự: [0]


### 3.4. `lru_cache`, duyệt ngược và `itertools.product`

- `@lru_cache(None)` nhớ kết quả hàm theo đối số, để mỗi `(i, bal)` chỉ tính một lần.
- `s[::-1]` là chuỗi đảo ngược; `reversed(s)` duyệt ngược không tạo chuỗi mới.
- `product("()_", repeat=k)` sinh mọi cách gán `k` dấu `*` (dùng `_` đại diện "rỗng"): đủ để làm **oracle** vét cạn cho chuỗi ngắn.

In [5]:
print(list(product("()_", repeat=2)))
print("(*))"[::-1], list(reversed("ab*")))

def expand(s: str, choice: str) -> str:
    """Thay lần lượt các '*' bằng các ký tự trong choice ('_' = rỗng)."""
    it = iter(choice)
    return "".join(("" if (r := next(it)) == "_" else r) if c == "*" else c for c in s)

print(expand("(*)*", "(_"), "|", expand("(*)*", ")("))

[('(', '('), ('(', ')'), ('(', '_'), (')', '('), (')', ')'), (')', '_'), ('_', '('), ('_', ')'), ('_', '_')]
))*( ['*', 'b', 'a']
(() | ())(


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Xét `"(*))"` với tập `S`:

| Sau ký tự | Quy tắc | `S` |
|---|---|---|
| (khởi đầu) | | `{0}` |
| `(` | `+1` | `{1}` |
| `*` | `-1, 0, +1` | `{0, 1, 2}` |
| `)` | `-1`, bỏ âm | `{0, 1}` |
| `)` | `-1`, bỏ âm | `{0}` |

Cuối cùng `0` thuộc `S` nên hợp lệ. Nhận xét quan trọng: **mỗi tập đều là một dãy số nguyên liên tiếp** (`{0,1,2}`, `{0,1}`, `{0}`). Vậy ta không cần cả tập; chỉ cần **cận dưới `lo`** và **cận trên `hi`**.

Hai ví dụ sai: `"("` cho `S = {1}`, không chứa 0 -> `false`. `"*("`: `S` sau `*` là `{0,1}` (bỏ `-1`), sau `(` là `{1, 2}` -> không chứa 0 -> `false`.

In [6]:
for s in ["(*))", "(", "*(", ")*"]:
    h = reachable_sets(s)
    print(f"{s!r:7}", [sorted(x) for x in h], "->", 0 in h[-1])

'(*))'  [[0], [1], [0, 1, 2], [0, 1], [0]] -> True
'('     [[0], [1]] -> False
'*('    [[0], [0, 1], [1, 2]] -> False
')*'    [[0], [], []] -> False


## 5. Cách 1 — Brute Force: đệ quy thử mọi lựa chọn cho `*`

**Ý tưởng.** Gọi `dfs(i, bal)` = "từ vị trí `i` với `bal` ngoặc mở đang chờ, có cách gán hợp lệ không?". Với `(` : `bal + 1`; với `)` : `bal - 1`; với `*`: thử **cả ba** lựa chọn. Nếu `bal` âm thì thất bại ngay; hết chuỗi thì thành công khi `bal == 0`.

In [7]:
def check_brute(s: str) -> bool:
    n = len(s)

    def dfs(i: int, bal: int) -> bool:
        if bal < 0:
            return False
        if i == n:
            return bal == 0
        c = s[i]
        if c == "(":
            return dfs(i + 1, bal + 1)
        if c == ")":
            return dfs(i + 1, bal - 1)
        return dfs(i + 1, bal + 1) or dfs(i + 1, bal - 1) or dfs(i + 1, bal)

    return dfs(0, 0)

for s, expected in examples:
    got = check_brute(s)
    print(s, got, "OK" if got == expected else "SAI")
    assert got == expected

() True OK
(*) True OK
(*)) True OK
( False OK


**Oracle độc lập.** Để kiểm mọi cách sau này ta có hai oracle không dùng ý tưởng của chúng:

1. **Vét cạn thay thế**: thử mọi cách gán `product("()_", repeat=k)` rồi kiểm tra bằng `valid_plain` (cho chuỗi ngắn).
2. **DP khoảng** `O(n^3)` (lời giải gốc của doocs): `dp[i][j]` = đoạn `s[i..j]` có thể hợp lệ không.

In [8]:
def oracle_product(s: str) -> bool:
    k = s.count("*")
    for choice in product("()_", repeat=k):
        if valid_plain(expand(s, "".join(choice))):
            return True
    return False

def oracle_interval(s: str) -> bool:
    n = len(s)
    dp = [[False] * n for _ in range(n)]
    for i, c in enumerate(s):
        dp[i][i] = c == "*"
    for i in range(n - 2, -1, -1):
        for j in range(i + 1, n):
            dp[i][j] = s[i] in "(*" and s[j] in "*)" and (i + 1 == j or dp[i + 1][j - 1])
            dp[i][j] = dp[i][j] or any(dp[i][k] and dp[k + 1][j] for k in range(i, j))
    return dp[0][-1]

for s, expected in examples:
    assert oracle_product(s) == oracle_interval(s) == expected
print("hai oracle khớp ví dụ đề")

hai oracle khớp ví dụ đề


**Độ phức tạp và nút thắt.** Mỗi `*` rẽ 3 nhánh nên xấu nhất `O(3^k)` với `k` là số dấu `*` (tới `3^100` khi `s` toàn `*`), bộ nhớ `O(n)` cho ngăn xếp đệ quy. Hãy đếm số lần gọi hàm để thấy bùng nổ. Nút thắt: **cùng một cặp `(i, bal)` bị tính đi tính lại** (nhiều đường dẫn khác nhau tới cùng một số dư).

In [9]:
def count_calls(s: str) -> int:
    n = len(s)
    cnt = 0
    def dfs(i, bal):
        nonlocal cnt
        cnt += 1
        if bal < 0:
            return False
        if i == n:
            return bal == 0
        c = s[i]
        if c == "(":
            return dfs(i + 1, bal + 1)
        if c == ")":
            return dfs(i + 1, bal - 1)
        return dfs(i + 1, bal + 1) or dfs(i + 1, bal - 1) or dfs(i + 1, bal)
    dfs(0, 0)
    return cnt

# chuỗi "xấu" buộc thử hết: toàn '*' rồi một ')' thừa đầu cuối làm thất bại muộn
print(f"{'k':>3} {'số lần gọi':>12} {'số trạng thái (i, bal) tối đa':>32}")
for k in (2, 4, 6, 8, 10, 12):
    s = "*" * k + "("          # '(' ở cuối khiến mọi nhánh thất bại ở lần kiểm cuối
    print(f"{k:>3} {count_calls(s):>12} {(k + 2) * (k + 3):>32}")

  k   số lần gọi    số trạng thái (i, bal) tối đa
  2           15                               20
  4           99                               42
  6          724                               72
  8         5631                              110
 10        45318                              156
 12       372452                              210


Số lần gọi nhân khoảng **3 lần** mỗi khi thêm một `*`, trong khi số trạng thái `(i, bal)` khác nhau chỉ tăng bậc hai. Có thể "nhớ" kết quả.

## 6. Ý tưởng thuật toán / bước đột phá

Hai bước đột phá nối tiếp:

1. **Ghi nhớ `(i, bal)`.** `bal` không vượt quá `n` nên có `O(n^2)` trạng thái. Đây là Cách 2 (DP / memo).
2. **Thu gọn tập thành khoảng `[lo, hi]`.** Ở mục 4 ta thấy tập số dư khả dĩ luôn là một khoảng liên tục. Lý do: bắt đầu `{0}`; `(` và `)` chỉ dịch cả tập; `*` biến tập liên tiếp `[a, b]` thành `[a-1, b+1]` (liên tục, vì có thêm lựa chọn "rỗng" nên không bao giờ bỏ lỡ số nào ở giữa); việc bỏ giá trị âm chỉ cắt đầu khoảng. Vậy chỉ cần hai số: `lo` = **số ngoặc mở ít nhất** có thể (coi mọi `*` là `)` nếu được), `hi` = **nhiều nhất** (coi mọi `*` là `(`).

Từ đó có luật cập nhật:

- `(`: `lo += 1`, `hi += 1`
- `)`: `lo -= 1`, `hi -= 1`
- `*`: `lo -= 1`, `hi += 1`
- Nếu `hi < 0`: **thất bại** (kể cả coi mọi `*` là `(` vẫn thiếu `(`).
- `lo = max(lo, 0)` (không cho số dư âm: ta có thể chọn `*` đó là rỗng).
- Cuối: hợp lệ nếu `lo == 0`.

Ngoài ra còn hai cách nhìn khác cùng đạt `O(n)`: **hai ngăn xếp chỉ số** (ghép `)` với `(` trước, rồi `*`; cuối cùng ghép `(` còn lại với `*` đứng sau) và **hai lượt quét** (trái -> phải coi `*` là `(`; phải -> trái coi `*` là `)`).

| Cách | Ý tưởng |
|---|---|
| 2 | Memo `(i, bal)` — trạng thái đầy đủ |
| 3 | Hai ngăn xếp chỉ số `(` và `*` |
| 4 | Tham lam: duy trì khoảng `[lo, hi]` |
| 5 | Hai lượt quét: `*` là `(`, rồi `*` là `)` |

In [10]:
# Kiểm chứng "tập luôn là khoảng liên tục" trên MỌI chuỗi độ dài <= 8
def all_strings(max_len: int, alphabet="()*"):
    out, frontier = [], [""]
    for _ in range(max_len):
        frontier = [x + c for x in frontier for c in alphabet]
        out += frontier
    return out

def interval_after(s):
    lo = hi = 0
    for c in s:
        if c == "(":
            lo, hi = lo + 1, hi + 1
        elif c == ")":
            lo, hi = lo - 1, hi - 1
        else:
            lo, hi = lo - 1, hi + 1
        if hi < 0:
            return None
        lo = max(lo, 0)
    return lo, hi

checked = 0
for s in all_strings(8):
    S = reachable_sets(s)[-1]
    iv = interval_after(s)
    if not S:
        assert iv is None, s
    else:
        assert iv is not None and S == set(range(iv[0], iv[1] + 1)), (s, S, iv)
    checked += 1
print("tập khả dĩ luôn bằng khoảng [lo, hi] (hoặc rỗng khi hi < 0) trên", checked, "chuỗi")

tập khả dĩ luôn bằng khoảng [lo, hi] (hoặc rỗng khi hi < 0) trên 9840 chuỗi


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy có nhớ theo `(i, bal)`

**Giải thích từng đoạn.**

- Giống Cách 1 nhưng bọc `dfs` bằng `@lru_cache(None)`.
- Thêm cắt tỉa: nếu `bal > n - i` (số `(` chờ đóng nhiều hơn số ký tự còn lại) thì chắc chắn thất bại.
- Trạng thái: `i` từ `0..n`, `bal` từ `0..n`, nên tối đa `(n + 1)^2` cặp.

In [11]:
def check_memo(s: str) -> bool:
    n = len(s)

    @lru_cache(None)
    def dfs(i: int, bal: int) -> bool:
        if bal < 0 or bal > n - i:
            return False
        if i == n:
            return bal == 0
        c = s[i]
        if c == "(":
            return dfs(i + 1, bal + 1)
        if c == ")":
            return dfs(i + 1, bal - 1)
        return dfs(i + 1, bal + 1) or dfs(i + 1, bal - 1) or dfs(i + 1, bal)

    return dfs(0, 0)

for s, expected in examples:
    assert check_memo(s) == expected
print("memo OK")

memo OK


**Code nộp LeetCode (Cách 2):**

In [12]:
class Solution:
    def checkValidString(self, s: str) -> bool:
        from functools import lru_cache
        n = len(s)

        @lru_cache(None)
        def dfs(i: int, bal: int) -> bool:
            if bal < 0 or bal > n - i:
                return False
            if i == n:
                return bal == 0
            c = s[i]
            if c == "(":
                return dfs(i + 1, bal + 1)
            if c == ")":
                return dfs(i + 1, bal - 1)
            return dfs(i + 1, bal + 1) or dfs(i + 1, bal - 1) or dfs(i + 1, bal)

        return dfs(0, 0)

sol_memo = Solution().checkValidString
print([sol_memo(s) for s, _ in examples])

[True, True, True, False]


**Độ phức tạp (Cách 2):** thời gian `O(n^2)` (mỗi trạng thái `O(1)` việc), bộ nhớ `O(n^2)` cho bộ nhớ đệm cộng `O(n)` ngăn xếp đệ quy (độ sâu tới `n = 100`, an toàn).

---

### 7.2. Cách 3 — Hai ngăn xếp chỉ số

**Ý tưởng.** Cất **chỉ số** của `(` vào `left` và của `*` vào `star`. Gặp `)`:

1. ưu tiên ghép với một `(` (lấy đỉnh `left`);
2. nếu không có `(`, ghép với một `*` (coi `*` là `(`);
3. nếu không có gì, thất bại (`)` thừa).

Cuối chuỗi còn các `(` chưa được ghép: mỗi `(` phải ghép với một `*` **đứng sau nó** (coi `*` là `)`). Ghép từ cặp đỉnh: lấy `(` đỉnh (chỉ số lớn nhất) và `*` đỉnh (chỉ số lớn nhất); nếu `chỉ số '(' > chỉ số '*'` thì `*` đứng trước `(`, không thể ghép -> thất bại. Hợp lệ khi mọi `(` được ghép.

**Giải thích từng đoạn.**

- Vì sao ưu tiên `(` thật hơn `*` khi gặp `)`: `*` linh hoạt hơn (có thể là rỗng hay `)`), nên để dành.
- Vì sao ghép cặp đỉnh ở cuối: các `(` còn lại và các `*` còn lại đều theo thứ tự chỉ số tăng dần; ghép lớn với lớn là cách dễ thỏa nhất ("`(` muộn nhất cần `*` muộn nhất").

In [13]:
def check_stacks(s: str) -> bool:
    left, star = [], []
    for i, c in enumerate(s):
        if c == "(":
            left.append(i)
        elif c == "*":
            star.append(i)
        else:
            if left:
                left.pop()
            elif star:
                star.pop()
            else:
                return False
    while left and star:
        if left.pop() > star.pop():
            return False
    return not left

for s, expected in examples:
    assert check_stacks(s) == expected
print("hai ngăn xếp OK")

hai ngăn xếp OK


**Code nộp LeetCode (Cách 3):**

In [14]:
class Solution:
    def checkValidString(self, s: str) -> bool:
        left, star = [], []
        for i, c in enumerate(s):
            if c == "(":
                left.append(i)
            elif c == "*":
                star.append(i)
            else:
                if left:
                    left.pop()
                elif star:
                    star.pop()
                else:
                    return False
        while left and star:
            if left.pop() > star.pop():
                return False
        return not left

sol_stacks = Solution().checkValidString
print([sol_stacks(s) for s, _ in examples])

[True, True, True, False]


**Độ phức tạp (Cách 3):** thời gian `O(n)` (mỗi ký tự đẩy/lấy tối đa một lần), bộ nhớ `O(n)` cho hai ngăn xếp.

---

### 7.3. Cách 4 — Tham lam khoảng `[lo, hi]`

**Giải thích từng đoạn.** (Luật cập nhật ở mục 6.)

- `lo`: số `(` chờ đóng **tối thiểu** (cố gắng coi `*` là `)`), nhưng không bao giờ dưới 0.
- `hi`: **tối đa** (coi `*` là `(`).
- Nếu `hi < 0` ở bất kỳ lúc nào: dù coi mọi `*` là `(` vẫn có `)` thừa -> `False` ngay.
- Cuối cùng cần `0` trong `[lo, hi]`; vì `lo >= 0` nên điều kiện là `lo == 0`.

In [15]:
def check_greedy(s: str) -> bool:
    lo = hi = 0
    for c in s:
        if c == "(":
            lo += 1
            hi += 1
        elif c == ")":
            lo -= 1
            hi -= 1
        else:
            lo -= 1
            hi += 1
        if hi < 0:
            return False
        lo = max(lo, 0)
    return lo == 0

for s, expected in examples:
    assert check_greedy(s) == expected
print("greedy OK")

greedy OK


**Code nộp LeetCode (Cách 4):**

In [16]:
class Solution:
    def checkValidString(self, s: str) -> bool:
        lo = hi = 0
        for c in s:
            if c == "(":
                lo += 1
                hi += 1
            elif c == ")":
                lo -= 1
                hi -= 1
            else:
                lo -= 1
                hi += 1
            if hi < 0:
                return False
            lo = max(lo, 0)
        return lo == 0

sol_greedy = Solution().checkValidString
print([sol_greedy(s) for s, _ in examples])

[True, True, True, False]


**Độ phức tạp (Cách 4):** thời gian `O(n)`, bộ nhớ `O(1)`. **Đây là cách nên nộp.**

---

### 7.4. Cách 5 — Hai lượt quét (trái -> phải, phải -> trái)

**Ý tưởng.** Tách hai điều kiện:

1. **Không có `)` thừa:** quét **trái -> phải**, coi `*` là `(` (lợi nhất để chống `)` thừa). Biến `x` đếm "số `(` hoặc `*` đang có thể dùng để đóng". Gặp `)`: nếu `x == 0` -> thất bại, ngược lại `x -= 1`.
2. **Không có `(` thừa:** quét **phải -> trái**, coi `*` là `)`. Đếm `)` hoặc `*`; gặp `(` mà `x == 0` -> thất bại.

Cả hai lượt đều qua thì hợp lệ.

**Giải thích từng đoạn.**

- Lượt 1 chỉ đảm bảo mọi `)` có thứ gì phía trước để ghép (kể cả `*`), chưa quan tâm `(` thừa.
- Lượt 2 đảm bảo mọi `(` có thứ gì phía sau để ghép.
- Hai điều kiện cần; ta chứng minh ở mục 10 rằng cùng nhau chúng đủ.

In [17]:
def check_two_pass(s: str) -> bool:
    x = 0
    for c in s:
        if c in "(*":
            x += 1
        elif x:
            x -= 1
        else:
            return False
    x = 0
    for c in reversed(s):
        if c in "*)":
            x += 1
        elif x:
            x -= 1
        else:
            return False
    return True

for s, expected in examples:
    assert check_two_pass(s) == expected
print("hai lượt OK")

hai lượt OK


**Code nộp LeetCode (Cách 5):**

In [18]:
class Solution:
    def checkValidString(self, s: str) -> bool:
        x = 0
        for c in s:
            if c in "(*":
                x += 1
            elif x:
                x -= 1
            else:
                return False
        x = 0
        for c in s[::-1]:
            if c in "*)":
                x += 1
            elif x:
                x -= 1
            else:
                return False
        return True

sol_two_pass = Solution().checkValidString
print([sol_two_pass(s) for s, _ in examples])

[True, True, True, False]


**Độ phức tạp (Cách 5):** thời gian `O(n)` (hai lượt), bộ nhớ `O(1)` (`s[::-1]` tạo bản sao `O(n)`; dùng `reversed(s)` để giữ `O(1)`).

## 8. TRACE từng bước

### 8.1. Cách 4 với `"(*))"`

| Ký tự | `lo`, `hi` sau cập nhật | `hi < 0`? | `lo` sau `max(lo, 0)` |
|---|---|---|---|
| (khởi đầu) | 0, 0 | | 0 |
| `(` | 1, 1 | không | 1 |
| `*` | 0, 2 | không | 0 |
| `)` | -1, 1 | không | 0 |
| `)` | -1, 0 | không | 0 |

Cuối cùng `lo == 0` -> `True`. Hàm `trace_greedy` in từng bước.

In [19]:
def trace_greedy(s: str) -> bool:
    lo = hi = 0
    print(f"s = {s!r}; bắt đầu lo=0, hi=0")
    for i, c in enumerate(s):
        if c == "(":
            lo, hi = lo + 1, hi + 1
        elif c == ")":
            lo, hi = lo - 1, hi - 1
        else:
            lo, hi = lo - 1, hi + 1
        note = ""
        if hi < 0:
            print(f"  i={i} {c!r}: lo={lo}, hi={hi} -> hi < 0: không còn cách nào, False")
            return False
        if lo < 0:
            note = f" (lo âm -> đặt về 0)"
            lo = 0
        print(f"  i={i} {c!r}: lo={lo}, hi={hi}{note}")
    print(f"  kết thúc: lo={lo} -> {'0 thuộc [lo, hi]' if lo == 0 else 'không có cách dùng hết ngoặc mở'}")
    return lo == 0

assert trace_greedy("(*))") is True
print()
assert trace_greedy("(((*") is False
print()
assert trace_greedy(")(*") is False

s = '(*))'; bắt đầu lo=0, hi=0
  i=0 '(': lo=1, hi=1
  i=1 '*': lo=0, hi=2
  i=2 ')': lo=0, hi=1 (lo âm -> đặt về 0)
  i=3 ')': lo=0, hi=0 (lo âm -> đặt về 0)
  kết thúc: lo=0 -> 0 thuộc [lo, hi]

s = '(((*'; bắt đầu lo=0, hi=0
  i=0 '(': lo=1, hi=1
  i=1 '(': lo=2, hi=2
  i=2 '(': lo=3, hi=3
  i=3 '*': lo=2, hi=4
  kết thúc: lo=2 -> không có cách dùng hết ngoặc mở

s = ')(*'; bắt đầu lo=0, hi=0
  i=0 ')': lo=-1, hi=-1 -> hi < 0: không còn cách nào, False


### 8.2. Cách 3 (hai ngăn xếp) với `"(*()"`

Xem cách ghép `)` rồi ghép cuối chuỗi. Ví dụ `"(*()"`: chỉ số 0 `(`, 1 `*`, 2 `(`, 3 `)`.

| i | ký tự | `left` | `star` | hành động |
|---|---|---|---|---|
| 0 | `(` | [0] | [] | đẩy |
| 1 | `*` | [0] | [1] | đẩy |
| 2 | `(` | [0, 2] | [1] | đẩy |
| 3 | `)` | [0] | [1] | ghép với `(` ở 2 |

Cuối: `left = [0]`, `star = [1]`: `0 < 1` nên `(` ở 0 ghép được với `*` ở 1 (coi `*` là `)`) -> `True`.

In [20]:
def trace_stacks(s: str) -> bool:
    left, star = [], []
    print(f"s = {s!r}")
    for i, c in enumerate(s):
        if c == "(":
            left.append(i); act = "đẩy vào left"
        elif c == "*":
            star.append(i); act = "đẩy vào star"
        else:
            if left:
                j = left.pop(); act = f"')' ghép với '(' ở {j}"
            elif star:
                j = star.pop(); act = f"')' ghép với '*' ở {j} (coi * là '(')"
            else:
                print(f"  i={i} ')' không còn gì để ghép -> False")
                return False
        print(f"  i={i} {c!r}: left={left} star={star}  ({act})")
    while left and star:
        l, st = left.pop(), star.pop()
        ok = l < st
        print(f"  cuối: ghép '(' ở {l} với '*' ở {st}: {'được ( ( đứng trước * )' if ok else 'KHÔNG (* đứng trước ()'}")
        if not ok:
            return False
    print(f"  còn dư '(' chưa ghép: {left}")
    return not left

assert trace_stacks("(*()") is True
print()
assert trace_stacks("*(") is False
print()
assert trace_stacks("(((*") is False

s = '(*()'
  i=0 '(': left=[0] star=[]  (đẩy vào left)
  i=1 '*': left=[0] star=[1]  (đẩy vào star)
  i=2 '(': left=[0, 2] star=[1]  (đẩy vào left)
  i=3 ')': left=[0] star=[1]  (')' ghép với '(' ở 2)
  cuối: ghép '(' ở 0 với '*' ở 1: được ( ( đứng trước * )
  còn dư '(' chưa ghép: []

s = '*('
  i=0 '*': left=[] star=[0]  (đẩy vào star)
  i=1 '(': left=[1] star=[0]  (đẩy vào left)
  cuối: ghép '(' ở 1 với '*' ở 0: KHÔNG (* đứng trước ()

s = '(((*'
  i=0 '(': left=[0] star=[]  (đẩy vào left)
  i=1 '(': left=[0, 1] star=[]  (đẩy vào left)
  i=2 '(': left=[0, 1, 2] star=[]  (đẩy vào left)
  i=3 '*': left=[0, 1, 2] star=[3]  (đẩy vào star)
  cuối: ghép '(' ở 2 với '*' ở 3: được ( ( đứng trước * )
  còn dư '(' chưa ghép: [0, 1]


### 8.3. Cách 5 (hai lượt) với `"(*))"` và `"*("`

Lượt trái -> phải coi `*` là `(`; lượt phải -> trái coi `*` là `)`.

In [21]:
def trace_two_pass(s: str) -> bool:
    print(f"s = {s!r}")
    x = 0
    print("  lượt 1 (trái -> phải, '*' như '('):")
    for i, c in enumerate(s):
        if c in "(*":
            x += 1
        elif x:
            x -= 1
        else:
            print(f"    i={i} {c!r}: x = 0 mà gặp ')' -> False")
            return False
        print(f"    i={i} {c!r}: x={x}")
    x = 0
    print("  lượt 2 (phải -> trái, '*' như ')'):")
    for i in range(len(s) - 1, -1, -1):
        c = s[i]
        if c in "*)":
            x += 1
        elif x:
            x -= 1
        else:
            print(f"    i={i} {c!r}: x = 0 mà gặp '(' -> False")
            return False
        print(f"    i={i} {c!r}: x={x}")
    return True

assert trace_two_pass("(*))") is True
print()
assert trace_two_pass("*(") is False

s = '(*))'
  lượt 1 (trái -> phải, '*' như '('):
    i=0 '(': x=1
    i=1 '*': x=2
    i=2 ')': x=1
    i=3 ')': x=0
  lượt 2 (phải -> trái, '*' như ')'):
    i=3 ')': x=1
    i=2 ')': x=2
    i=1 '*': x=3
    i=0 '(': x=2

s = '*('
  lượt 1 (trái -> phải, '*' như '('):
    i=0 '*': x=1
    i=1 '(': x=2
  lượt 2 (phải -> trái, '*' như ')'):
    i=1 '(': x = 0 mà gặp '(' -> False


## 9. Code nộp LeetCode

Ta chọn **Cách 4** (tham lam khoảng `[lo, hi]`): `O(n)` thời gian, `O(1)` bộ nhớ, ít chỗ sai khi đã hiểu hai luật `hi < 0` và `lo = max(lo, 0)`.

```python
class Solution:
    def checkValidString(self, s: str) -> bool:
        lo = hi = 0
        for c in s:
            if c == "(":
                lo += 1
                hi += 1
            elif c == ")":
                lo -= 1
                hi -= 1
            else:
                lo -= 1
                hi += 1
            if hi < 0:
                return False
            lo = max(lo, 0)
        return lo == 0
```

In [22]:
class Solution:
    def checkValidString(self, s: str) -> bool:
        lo = hi = 0
        for c in s:
            if c == "(":
                lo += 1
                hi += 1
            elif c == ")":
                lo -= 1
                hi -= 1
            else:
                lo -= 1
                hi += 1
            if hi < 0:
                return False
            lo = max(lo, 0)
        return lo == 0

sol = Solution()
for s, expected in examples:
    got = sol.checkValidString(s)
    print(f"{s!r:8} ->", got, "OK" if got == expected else "SAI")
    assert got == expected

'()'     -> True OK
'(*)'    -> True OK
'(*))'   -> True OK
'('      -> False OK


## 10. Vì sao đúng

**(a) Tập số dư là khoảng liên tục (Cách 4).** Gọi `S_t` là tập `bal` khả dĩ sau `t` ký tự (đã loại giá trị âm). `S_0 = {0}`. Giả sử `S_t = [a, b]` (liên tục). `(` cho `[a+1, b+1]`; `)` cho `[a-1, b-1]` rồi cắt phần âm; `*` cho `[a-1, b+1]` (hợp của ba khoảng dịch `[a-1,b-1] ∪ [a,b] ∪ [a+1,b+1]` là liên tục vì chúng chồng nhau) rồi cắt phần âm. Vậy `S_{t+1}` vẫn liên tục (hoặc rỗng khi `b + delta < 0`, tương ứng điều kiện `hi < 0`). Do đó `(lo, hi)` mô tả **chính xác** `S_t`, và hợp lệ khi `0 ∈ S_n` tức `lo == 0` (vì `lo >= 0` luôn đúng sau khi cắt). Mục 6 đã kiểm bằng máy trên mọi chuỗi độ dài `<= 8`.

**(b) Cách 2** chính là mô phỏng trực tiếp định nghĩa hợp lệ (có một cách gán sao cho `bal` không âm và cuối bằng 0); ghi nhớ không đổi kết quả.

**(c) Cách 5 (hai lượt) tương đương Cách 4.**

- Lượt 1: biến `x` chính là `hi` (cộng 1 cho `(` và `*`, trừ 1 cho `)`), và lượt 1 thất bại đúng lúc `hi < 0`. Vậy lượt 1 qua khi và chỉ khi Cách 4 không bao giờ dừng vì `hi < 0`.
- Lượt 2: duyệt ngược, `x` = (số `)` và `*`) - (số `(`) của phần đuôi đã duyệt; lượt 2 thất bại khi có một **đuôi** của chuỗi mà số `(` nhiều hơn số `)` cộng số `*`. Mặt khác `lo = max(lo + d, 0)` với `d = +1` cho `(`, `-1` cho `)` và `*` là một dạng đệ quy "cận 0" nên `lo` cuối cùng bằng `max(0, max trên mọi đuôi của (số '(' - số ')' - số '*'))`. Vậy `lo == 0` khi và chỉ khi không đuôi nào có `(` nhiều hơn `)` + `*`, tức lượt 2 qua.

Cách 4 đúng theo (a) nên Cách 5 cũng đúng.

**(d) Cách 3.** Ưu tiên ghép `)` với `(` thật vì `*` linh hoạt hơn (còn dùng được làm rỗng/`)`/`(`); nếu thay `(` thật bằng `*` thì không bao giờ tốt hơn. Cuối chuỗi, mỗi `(` còn lại cần một `*` sau nó làm `)`: ghép "lớn với lớn" là cách thuận lợi nhất (thuật toán ghép tham lam trên hai dãy được sắp thứ tự).

**Kiểm chứng bằng máy:** so cả 5 cách với hai oracle (vét cạn `product` và DP khoảng) trên **mọi** chuỗi độ dài `<= 8` (9840 chuỗi), chính xác hơn mọi ví dụ ngẫu nhiên.

In [23]:
solutions = {
    "brute": check_brute,
    "memo": check_memo,
    "stacks": check_stacks,
    "greedy": check_greedy,
    "two_pass": check_two_pass,
    "sol_memo": sol_memo,
    "sol_stacks": sol_stacks,
    "sol_greedy": sol_greedy,
    "sol_two_pass": sol_two_pass,
    "final": Solution().checkValidString,
}

total = 0
n_true = 0
for s in all_strings(8):
    ref = oracle_product(s)
    assert oracle_interval(s) == ref, s
    for name, fn in solutions.items():
        assert fn(s) == ref, (name, s, ref)
    total += 1
    n_true += ref
print(f"mọi chuỗi độ dài 1..8: {total} chuỗi, {n_true} hợp lệ, {total - n_true} không hợp lệ; {len(solutions)} cài đặt khớp 2 oracle")

mọi chuỗi độ dài 1..8: 9840 chuỗi, 2813 hợp lệ, 7027 không hợp lệ; 10 cài đặt khớp 2 oracle


## 11. Time / Space Complexity

Gọi `n = len(s)` (`<= 100`), `k` là số dấu `*`.

| Cách | Time | Space |
|---|---|---|
| 1. Đệ quy thuần | `O(3^k)` (xấu nhất `O(3^n)`) | `O(n)` |
| 2. Đệ quy có nhớ `(i, bal)` | `O(n^2)` | `O(n^2)` |
| 3. Hai ngăn xếp | `O(n)` | `O(n)` |
| 4. Khoảng `[lo, hi]` | `O(n)` | `O(1)` |
| 5. Hai lượt quét | `O(n)` | `O(1)` (`reversed`) |
| Oracle DP khoảng | `O(n^3)` | `O(n^2)` |

Với `n = 100` các cách `O(n)` chạy dưới một mili-giây, `O(n^2)` vài mili-giây, DP khoảng `O(n^3)` (mười lăm triệu phép) chậm hơn nhiều. Đo thử:

In [24]:
random.seed(678)
big = ["".join(random.choice("()*") for _ in range(100)) for _ in range(200)]
big += ["(" * 50 + "*" * 50, "*" * 100, "(" * 100, "*" * 50 + "(" * 50]
for name in ["memo", "stacks", "greedy", "two_pass"]:
    fn = solutions[name]
    t0 = time.perf_counter()
    r = [fn(s) for s in big]
    dt = (time.perf_counter() - t0) * 1000
    print(f"{name:9} 204 chuỗi dài 100: {dt:8.2f} ms")
sample = big[:12] + big[-4:]
t0 = time.perf_counter()
ref = [oracle_interval(s) for s in sample]
print(f"{'interval':9} {len(sample)} chuỗi dài 100: {(time.perf_counter() - t0) * 1000:8.2f} ms (oracle O(n^3))")
for name in ["memo", "stacks", "greedy", "two_pass"]:
    assert [solutions[name](s) for s in sample] == ref, name
print("khớp oracle DP khoảng trên chuỗi dài 100")

memo      204 chuỗi dài 100:    55.32 ms
stacks    204 chuỗi dài 100:     0.81 ms
greedy    204 chuỗi dài 100:     1.06 ms
two_pass  204 chuỗi dài 100:     0.80 ms
interval  16 chuỗi dài 100:   102.65 ms (oracle O(n^3))
khớp oracle DP khoảng trên chuỗi dài 100


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay, phản ví dụ cho các cách sai, đối chiếu mọi cách với oracle trên input ngẫu nhiên có seed (đúng ràng buộc độ dài `<= 100`), và cả chuỗi cực trị.

In [25]:
# 12.1 ví dụ đề
for s, expected in examples:
    for name, fn in solutions.items():
        assert fn(s) == expected, (name, s)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay)
edge = [
    ("*", True),
    ("(", False),
    (")", False),
    ("()", True),
    ("(*", True),            # * là ')'
    ("*)", True),            # * là '('
    (")*", False),           # ')' đầu tiên không có '(' trước
    ("*(", False),           # '(' cuối không có gì sau
    ("*(*", True),           # '(' ghép với '*' cuối
    ("(((*", False),         # 3 '(' chỉ có 1 '*'
    ("((**", True),
    ("(*)))", False),
    ("((*)", True),
    ("******", True),
    ("(*()", True),
    ("()()", True),
    (")(", False),
    ("**((", False),
    ("(" * 50 + "*" * 50, True),
    ("*" * 50 + "(" * 50, False),
    ("(" * 100, False),
    ("*" * 100, True),
    ("(" * 50 + ")" * 50, True),
    ("(" * 49 + "*" + ")" * 50, True),
]
for s, expected in edge:
    if len(s) <= 30:
        assert oracle_interval(s) == expected, ("oracle khác kỳ vọng", s)
    for name, fn in solutions.items():
        if name == "brute" and len(s) > 14:
            continue                      # 3^k quá lớn với chuỗi dài: chính là lý do cần cách tốt hơn
        assert fn(s) == expected, (name, s, expected)
print("12.2 edge case: OK", len(edge), "ca")

# 12.3 phản ví dụ: các cách sai
def wrong_totals(s):          # chỉ so tổng số ký tự, bỏ qua thứ tự
    o, c, st = s.count("("), s.count(")"), s.count("*")
    return o + st >= c and c + st >= o

def wrong_no_clip(s):         # quên lo = max(lo, 0), kiểm cuối bằng lo <= 0 <= hi
    lo = hi = 0
    for ch in s:
        if ch == "(":
            lo += 1; hi += 1
        elif ch == ")":
            lo -= 1; hi -= 1
        else:
            lo -= 1; hi += 1
        if hi < 0:
            return False
    return lo <= 0 <= hi

def wrong_no_hi_check(s):     # quên dừng khi hi < 0
    lo = hi = 0
    for ch in s:
        if ch == "(":
            lo += 1; hi += 1
        elif ch == ")":
            lo -= 1; hi -= 1
        else:
            lo -= 1; hi += 1
        lo = max(lo, 0)
    return lo == 0

def wrong_stack_order(s):     # ghép '(' với '*' mà bỏ qua thứ tự
    left, star = [], []
    for i, c in enumerate(s):
        if c == "(":
            left.append(i)
        elif c == "*":
            star.append(i)
        else:
            if left:
                left.pop()
            elif star:
                star.pop()
            else:
                return False
    return len(left) <= len(star)

def wrong_one_pass(s):        # chỉ chạy lượt trái -> phải
    x = 0
    for c in s:
        if c in "(*":
            x += 1
        elif x:
            x -= 1
        else:
            return False
    return True

wrongs = [
    ("tổng ký tự", wrong_totals, "*("),
    ("quên chặn lo >= 0", wrong_no_clip, "*)("),
    ("quên dừng khi hi < 0", wrong_no_hi_check, ")(*"),
    ("ghép bỏ qua thứ tự", wrong_stack_order, "*("),
    ("chỉ một lượt quét", wrong_one_pass, "("),
]
for name, fn, bad in wrongs:
    truth = oracle_interval(bad)
    print(f"{name:22} trên {bad!r:6}: sai = {fn(bad)}, đúng = {truth}")
    assert fn(bad) != truth
print("12.3 phản ví dụ: OK (các cách sai đều bị lộ)")

# 12.4 ngẫu nhiên có seed, đúng ràng buộc (độ dài 1..100), nhiều tỷ lệ ký tự
rng = random.Random(2078)
cnt_true = 0
for _ in range(1500):
    n = rng.randint(1, 100)
    weights = rng.choice([(1, 1, 1), (3, 3, 1), (1, 1, 6), (5, 2, 2), (2, 5, 2)])
    s = "".join(rng.choices("()*", weights=weights, k=n))
    ref = oracle_interval(s) if n <= 40 else check_greedy(s)
    if n > 40:
        assert check_stacks(s) == check_two_pass(s) == check_memo(s) == ref
    for name in ("memo", "stacks", "greedy", "two_pass", "final"):
        assert solutions[name](s) == ref, (name, s)
    cnt_true += ref
print(f"12.4 ngẫu nhiên 1500 chuỗi: OK ({cnt_true} hợp lệ, {1500 - cnt_true} không)")

# 12.5 chuỗi được xây chắc chắn hợp lệ (đổi ngẫu nhiên một số ký tự của chuỗi ngoặc đúng thành '*')
rng = random.Random(9)
def random_valid(n_pairs):
    s, bal = [], 0
    opens = n_pairs
    while opens or bal:
        if opens and (bal == 0 or rng.random() < 0.5):
            s.append("("); opens -= 1; bal += 1
        else:
            s.append(")"); bal -= 1
    return "".join(s)

for _ in range(500):
    base = random_valid(rng.randint(1, 50))
    assert valid_plain(base)
    s = "".join("*" if rng.random() < 0.3 else c for c in base)
    for name in ("memo", "stacks", "greedy", "two_pass", "final"):
        assert solutions[name](s) is True, (name, s)
print("12.5 chuỗi hợp lệ bị thay một phần bằng '*' vẫn hợp lệ: OK")

12.1 ví dụ đề: OK
12.2 edge case: OK 24 ca
tổng ký tự             trên '*('  : sai = True, đúng = False
quên chặn lo >= 0      trên '*)(' : sai = True, đúng = False
quên dừng khi hi < 0   trên ')(*' : sai = True, đúng = False
ghép bỏ qua thứ tự     trên '*('  : sai = True, đúng = False
chỉ một lượt quét      trên '('   : sai = True, đúng = False
12.3 phản ví dụ: OK (các cách sai đều bị lộ)


12.4 ngẫu nhiên 1500 chuỗi: OK (354 hợp lệ, 1146 không)
12.5 chuỗi hợp lệ bị thay một phần bằng '*' vẫn hợp lệ: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ so tổng số ký tự

Nghĩ rằng chỉ cần `số '(' + số '*' >= số ')'` và `số ')' + số '*' >= số '('`. Thứ tự bị bỏ qua.

In [26]:
for s in ["*(", "(*)(", "*(("]:
    print(f"{s!r:7} cách sai = {wrong_totals(s)}, đáp án thật = {oracle_interval(s)}")
assert wrong_totals("*(") is True and oracle_interval("*(") is False

'*('    cách sai = True, đáp án thật = False
'(*)('  cách sai = True, đáp án thật = False
'*(('   cách sai = False, đáp án thật = False


Vì sao sai: `"*("` có `1` `(` và `1` `*` nhưng `(` đứng **cuối**, không có gì sau nó để đóng. Chuỗi ngoặc cần cả **thứ tự**. **Đúng:** dùng `lo/hi`, hai lượt hoặc hai ngăn xếp có chỉ số.

### Lỗi 2 — Quên chặn `lo >= 0`

Nếu để `lo` âm rồi kiểm cuối bằng `lo <= 0 <= hi`, thì một `*` thừa ở đầu làm `lo` âm vĩnh viễn và "che giấu" `(` thừa về sau.

In [27]:
s = "*)("
print("wrong_no_clip:", wrong_no_clip(s), "| đúng:", oracle_interval(s), "| code nộp:", Solution().checkValidString(s))
assert wrong_no_clip(s) is True and Solution().checkValidString(s) is False

wrong_no_clip: True | đúng: False | code nộp: False


Vì sao sai: `lo` là "số `(` chờ đóng ít nhất có thể", một số đếm **không thể âm** (ta luôn có thể chọn `*` là rỗng). **Đúng:** sau mỗi bước `lo = max(lo, 0)`.

### Lỗi 3 — Quên dừng khi `hi < 0`

`hi < 0` nghĩa là dù coi mọi `*` là `(` vẫn có `)` thừa. Nếu bỏ qua mà chỉ chặn `lo`, trạng thái "đi tiếp" sẽ vô nghĩa (thậm chí `lo > hi`).

In [28]:
s = ")(*"
print("wrong_no_hi_check:", wrong_no_hi_check(s), "| đúng:", oracle_interval(s))
assert wrong_no_hi_check(s) is True and Solution().checkValidString(s) is False

wrong_no_hi_check: True | đúng: False


Vì sao sai: `")"` ở đầu chuỗi không thể ghép với gì, đáng ra phải trả `False` ngay; nếu cứ đi tiếp, `(` và `*` phía sau làm `lo` về 0 một cách "oan". **Đúng:** `if hi < 0: return False`.

### Lỗi 4 — Ghép `(` với `*` mà không xét thứ tự

Ở Cách 3, cuối chuỗi chỉ so **số lượng** `len(left) <= len(star)`. Nhưng `*` phải đứng **sau** `(` mới làm `)` đóng được.

In [29]:
for s in ["*(", "**(", "*(*"]:
    print(f"{s!r:6} cách sai = {wrong_stack_order(s)}, đáp án thật = {oracle_interval(s)}")
assert wrong_stack_order("*(") is True and wrong_stack_order("**(") is True and oracle_interval("**(") is False

'*('   cách sai = True, đáp án thật = False
'**('  cách sai = True, đáp án thật = False
'*(*'  cách sai = True, đáp án thật = True


Vì sao sai: trong `"**("` hai `*` đứng **trước** `(` nên không đóng được nó. **Đúng:** so chỉ số `left.pop() < star.pop()`.

### Lỗi 5 — Chỉ chạy một lượt quét

Lượt trái -> phải chỉ phát hiện `)` thừa; không phát hiện `(` thừa. Cần thêm lượt phải -> trái (hoặc kiểm `lo == 0` ở cuối trong Cách 4).

In [30]:
for s in ["(", "((*", "(*("]:
    print(f"{s!r:6} một lượt = {wrong_one_pass(s)}, hai lượt = {check_two_pass(s)}, thật = {oracle_interval(s)}")
assert wrong_one_pass("(") is True and check_two_pass("(") is False

'('    một lượt = True, hai lượt = False, thật = False
'((*'  một lượt = True, hai lượt = False, thật = False
'(*('  một lượt = True, hai lượt = False, thật = False


Vì sao sai: `"("` qua lượt 1 vì không có `)` nào, nhưng `(` chưa được đóng. **Đúng:** lượt thứ hai (phải -> trái) bắt lỗi này.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** chuỗi ngoặc, ký tự "joker" có nhiều nghĩa, hỏi có **tồn tại** cách gán hợp lệ -> theo dõi **tập trạng thái**, hy vọng nó thu gọn được (khoảng).
- **Balance không âm + cuối bằng 0** là định nghĩa gọn của chuỗi ngoặc hợp lệ một loại.
- **`[lo, hi]` pattern:** khi tập trạng thái luôn liên tục thì lưu cận. Cũng gặp trong các bài "thay đổi +1/0/-1" khác.
- **Hai lượt (trái->phải rồi phải->trái)** thường xuất hiện khi hai ràng buộc đối xứng (`)` cần `(` trước, `(` cần `)` sau).
- Khi ghép bằng ngăn xếp, ưu tiên cặp **cứng nhắc** (`(` thật) trước, để dành ký tự linh hoạt (`*`) cho sau.
- `n <= 100` cho phép cả `O(n^3)` (DP khoảng) làm oracle, nhưng nên có lời giải `O(n)`.
- Test nên có: `"*("`, `")*"`, `"((*"`, chuỗi rất dài toàn `*`, toàn `(`.

## 15. Biến thể

1. **Valid Parentheses (20)**: nhiều loại ngoặc `()[]{}`, không có `*` -> ngăn xếp ký tự.
2. **Minimum Add to Make Parentheses Valid (921)**: tối thiểu số ngoặc cần thêm -> đếm `bal` và số `)` thừa.
3. **Longest Valid Parentheses (32)**, **Remove Invalid Parentheses (301)**: tương tự về ngữ cảnh nhưng khó hơn.
4. **Đếm số cách gán `*`** để hợp lệ (bài tập 5): DP theo `bal`.

Dưới đây là code cho biến thể 1 và 2, kiểm bằng oracle riêng.

In [31]:
def valid_multi(s: str) -> bool:
    pairs = {")": "(", "]": "[", "}": "{"}
    st = []
    for c in s:
        if c in "([{":
            st.append(c)
        elif not st or st.pop() != pairs[c]:
            return False
    return not st

def min_add_to_make_valid(s: str) -> int:
    bal = need = 0
    for c in s:
        if c == "(":
            bal += 1
        elif bal:
            bal -= 1
        else:
            need += 1          # ')' thừa phải thêm một '(' phía trước
    return bal + need

assert valid_multi("()[]{}") and valid_multi("{[()]}") and not valid_multi("(]") and not valid_multi("([)]") and not valid_multi("((")
assert min_add_to_make_valid("())") == 1 and min_add_to_make_valid("(((") == 3 and min_add_to_make_valid("()") == 0

# oracle BFS cho 921: thêm tối thiểu ký tự '(' hoặc ')' vào bất kỳ vị trí
def min_add_oracle(s: str) -> int:
    from collections import deque
    seen, dq = {s}, deque([(s, 0)])
    while dq:
        t, d = dq.popleft()
        if valid_plain(t):
            return d
        for i in range(len(t) + 1):
            for c in "()":
                u = t[:i] + c + t[i:]
                if u not in seen:
                    seen.add(u); dq.append((u, d + 1))
    return -1

for t in all_strings(5, "()"):
    assert min_add_to_make_valid(t) == min_add_oracle(t), t
print("Biến thể 20 và 921: OK")

Biến thể 20 và 921: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đệ quy thuần | Thử 3 lựa chọn mỗi `*` | `O(3^k)` | `O(n)` | Hiểu đề; chuỗi rất ngắn |
| 2. Memo `(i, bal)` | Trạng thái đầy đủ | `O(n^2)` | `O(n^2)` | Dễ suy ra từ đệ quy; khi chưa thấy tính liên tục |
| 3. Hai ngăn xếp | Ghép `)` trước, rồi `(`-`*` theo thứ tự | `O(n)` | `O(n)` | Cần biết **cặp** nào ghép với nhau |
| 4. Khoảng `[lo, hi]` | Tập số dư liên tục | `O(n)` | `O(1)` | **Cách nộp chính** |
| 5. Hai lượt quét | `*` là `(`, rồi `*` là `)` | `O(n)` | `O(1)` | Ngắn nhất; khi đã tin tính đối xứng |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng sau (dùng hàm `check_greedy` ở mục 7), rồi chạy để kiểm tra:

```python
print(check_greedy("(*)"))
print(check_greedy("*("))
print(interval_after("(**"))
```

In [32]:
# Bài 1: ghi dự đoán của bạn (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `s = "(*))"`, ghi lại cặp `(lo, hi)` **sau mỗi ký tự** (đã áp `lo = max(lo, 0)`) dưới dạng list 4 phần tử, và kết quả cuối `True/False`.

In [33]:
# Bài 2: điền kết quả tính tay
hand_pairs = None     # [(lo, hi), ...] cho 4 ký tự
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm theo Cách 4.

In [34]:
def exercise_3_greedy(s: str) -> bool:
    lo = hi = 0
    for c in s:
        # TODO 1: cập nhật lo, hi theo từng loại ký tự
        pass
        # TODO 2: nếu hi < 0 thì trả False
        # TODO 3: lo = max(lo, 0)
    # TODO 4: trả về lo == 0
    return None


# assert exercise_3_greedy("()") is True
# assert exercise_3_greedy("(*))") is True
# assert exercise_3_greedy("(") is False
# assert exercise_3_greedy("*(") is False

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 5, hãy tự viết `exercise_4_two_pass(s)`: quét hai lượt (trái -> phải coi `*` là `(`; phải -> trái coi `*` là `)`).

In [35]:
def exercise_4_two_pass(s: str) -> bool:
    # TODO
    pass


# assert exercise_4_two_pass("(*)") is True
# assert exercise_4_two_pass("(*))") is True
# assert exercise_4_two_pass("(") is False
# assert exercise_4_two_pass("*(") is False

### Bài 5 — Mini challenge: đếm số cách gán `*` (khó)

Cho `s` chỉ có `(`, `)`, `*`. Đếm **số cách** gán mỗi `*` thành một trong ba lựa chọn (`(`, `)` hoặc rỗng) sao cho chuỗi kết quả hợp lệ. Hai cách gán khác nhau nếu có ít nhất một `*` được gán khác nhau.

```python
exercise_5_count_ways("*")     -> 1      (chỉ gán rỗng)
exercise_5_count_ways("**")    -> 2      (rỗng+rỗng, hoặc "(" rồi ")")
exercise_5_count_ways("(*)")   -> 1
exercise_5_count_ways("(**)")  -> 3
exercise_5_count_ways("(")     -> 0
```
Gợi ý: DP theo số dư `bal`: giữ một `dict` `bal -> số cách`.

In [36]:
def exercise_5_count_ways(s: str) -> int:
    # TODO
    pass


# assert exercise_5_count_ways("*") == 1
# assert exercise_5_count_ways("**") == 2
# assert exercise_5_count_ways("(*)") == 1
# assert exercise_5_count_ways("(**)") == 3
# assert exercise_5_count_ways("(") == 0

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `"(*)"` đã thấy ở ví dụ đề.
- Mức 2: `"*("`: sau `*` có `lo = 0` (đã chặn), `hi = 1`; sau `(` có `lo = 1`.
- Mức 3: `True`, `False`, và `interval_after("(**")` = `(0, 3)` (`(`: (1,1); `*`: (0,2); `*`: (-1,3) rồi chặn thành (0,3)).

**Bài 2**
- Mức 1: bắt đầu `(0, 0)`.
- Mức 2: `(` -> `(1, 1)`; `*` -> `(0, 2)`.
- Mức 3: `)` -> `(-1, 1)` rồi chặn `(0, 1)`; `)` -> `(-1, 0)` rồi chặn `(0, 0)`. Kết quả `True`.

**Bài 3**
- Mức 1: ba nhánh `if c == "(" ... elif c == ")" ... else`.
- Mức 2: `*` làm `lo -= 1`, `hi += 1`.
- Mức 3: sau cập nhật: `if hi < 0: return False`; `lo = max(lo, 0)`; cuối `return lo == 0`.

**Bài 4**
- Mức 1: lượt 1 dùng biến đếm `x` cho `(` và `*`.
- Mức 2: gặp `)` mà `x == 0` thì `return False`, ngược lại `x -= 1`.
- Mức 3: lượt 2 duyệt ngược, đếm `*` và `)`; gặp `(` mà `x == 0` thì `return False`; cuối `return True`.

**Bài 5**
- Mức 1: `ways = {0: 1}` ban đầu.
- Mức 2: `(`: dịch `+1`; `)`: dịch `-1` (bỏ âm); `*`: cộng ba dịch `-1, 0, +1` (bỏ âm).
- Mức 3: đáp án `ways.get(0, 0)`; kiểm tra với `product("()_", repeat=k)`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [37]:
print(check_greedy("(*)"))
print(check_greedy("*("))
print(interval_after("(**"))

assert check_greedy("(*)") is True
assert check_greedy("*(") is False
assert interval_after("(**") == (0, 3)

True
False
(0, 3)


### Đáp án Bài 2

| Ký tự | `(lo, hi)` |
|---|---|
| `(` | (1, 1) |
| `*` | (0, 2) |
| `)` | (0, 1) |
| `)` | (0, 0) |

Kết quả cuối `lo == 0` -> `True`.

In [38]:
hand_pairs_solution = [(1, 1), (0, 2), (0, 1), (0, 0)]
hand_answer_solution = True

pairs = []
lo = hi = 0
for c in "(*))":
    if c == "(":
        lo, hi = lo + 1, hi + 1
    elif c == ")":
        lo, hi = lo - 1, hi - 1
    else:
        lo, hi = lo - 1, hi + 1
    lo = max(lo, 0)
    pairs.append((lo, hi))
assert pairs == hand_pairs_solution
assert (lo == 0) is hand_answer_solution is check_greedy("(*))")
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [39]:
def exercise_3_greedy_solution(s: str) -> bool:
    lo = hi = 0
    for c in s:
        if c == "(":
            lo += 1
            hi += 1
        elif c == ")":
            lo -= 1
            hi -= 1
        else:
            lo -= 1
            hi += 1
        if hi < 0:
            return False
        lo = max(lo, 0)
    return lo == 0

assert exercise_3_greedy_solution("()") is True
assert exercise_3_greedy_solution("(*))") is True
assert exercise_3_greedy_solution("(") is False
assert exercise_3_greedy_solution("*(") is False
for s in all_strings(7):
    assert exercise_3_greedy_solution(s) == oracle_interval(s), s
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [40]:
def exercise_4_two_pass_solution(s: str) -> bool:
    x = 0
    for c in s:
        if c in "(*":
            x += 1
        elif x:
            x -= 1
        else:
            return False
    x = 0
    for c in reversed(s):
        if c in "*)":
            x += 1
        elif x:
            x -= 1
        else:
            return False
    return True

assert exercise_4_two_pass_solution("(*)") is True
assert exercise_4_two_pass_solution("(*))") is True
assert exercise_4_two_pass_solution("(") is False
assert exercise_4_two_pass_solution("*(") is False
for s in all_strings(7):
    assert exercise_4_two_pass_solution(s) == oracle_interval(s), s
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [41]:
def exercise_5_count_ways_solution(s: str) -> int:
    ways = {0: 1}
    for c in s:
        nxt = {}
        if c == "(":
            deltas = (1,)
        elif c == ")":
            deltas = (-1,)
        else:
            deltas = (-1, 0, 1)
        for bal, cnt in ways.items():
            for d in deltas:
                nb = bal + d
                if nb >= 0:
                    nxt[nb] = nxt.get(nb, 0) + cnt
        ways = nxt
    return ways.get(0, 0)

def count_ways_oracle(s: str) -> int:
    k = s.count("*")
    return sum(valid_plain(expand(s, "".join(ch))) for ch in product("()_", repeat=k))

assert exercise_5_count_ways_solution("*") == 1
assert exercise_5_count_ways_solution("**") == 2
assert exercise_5_count_ways_solution("(*)") == 1
assert exercise_5_count_ways_solution("(**)") == 3
assert exercise_5_count_ways_solution("(") == 0
for s in all_strings(7):
    assert exercise_5_count_ways_solution(s) == count_ways_oracle(s), s
    assert (exercise_5_count_ways_solution(s) > 0) == check_greedy(s), s
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (khoảng `[lo, hi]`)

```python
class Solution:
    def checkValidString(self, s: str) -> bool:
        lo = hi = 0                 # số '(' chờ đóng: ít nhất / nhiều nhất
        for c in s:
            if c == "(":
                lo += 1; hi += 1
            elif c == ")":
                lo -= 1; hi -= 1
            else:                   # '*' : ')' , rỗng hoặc '('
                lo -= 1; hi += 1
            if hi < 0:              # dù * là '(' vẫn thừa ')'
                return False
            lo = max(lo, 0)         # có thể chọn * là rỗng nên không âm
        return lo == 0
```

Công thức nhớ nhanh: `)` thừa -> `hi < 0`; `(` thừa -> `lo > 0` khi kết thúc.

### Checklist trước khi submit

- [ ] `*` làm `lo -= 1` **và** `hi += 1`?
- [ ] Có `if hi < 0: return False` ngay trong vòng lặp?
- [ ] Có `lo = max(lo, 0)` sau mỗi ký tự?
- [ ] Cuối cùng kiểm `lo == 0` (không phải `hi == 0`)?
- [ ] Nếu dùng ngăn xếp: lưu **chỉ số** và so `chỉ số '(' < chỉ số '*'` khi ghép cuối?
- [ ] Nếu hai lượt: lượt 2 duyệt **ngược** và coi `*` như `)`?
- [ ] Đã thử `"*("`, `")*"`, `"(((*"`, `"*" * 100`, `"(" * 100`?
- [ ] Tên method đúng `checkValidString(self, s)` và trả `bool`?